# Chapter 6: Asynchronous Messaging Patterns

Synchronous calls create **tight coupling** — caller and callee must both be available.
Event-driven messaging decouples producers from consumers across time and space.

**Concepts:**
- Events vs Commands (naming contract)
- AgentEventBus with idempotency and dead-letter queue
- Causal tracing with EventContext
- Azure Service Bus integration

In [ ]:
import sys, os, asyncio
sys.path.insert(0, os.path.abspath('../..'))
from core.messaging.events import BaseMessage, Command, Event, EventContext
from core.messaging.bus import AgentEventBus, DeadLetterQueue
print('Messaging module ready.')

## 6.1 The Naming Contract: Events vs Commands

| Type | Grammar | Immutable? | Can be rejected? |
|------|---------|-----------|-----------------|
| **Event** | Past participle: `OrderPlaced`, `PaymentProcessed` | Yes | No |
| **Command** | Imperative verb: `ProcessOrder`, `RefundPayment` | No | Yes |

Mixing these creates bugs: an event named `ProcessOrder` implies a command hidden
in an event envelope, which breaks the immutability guarantee.

In [ ]:
import uuid

# Correct: Events describe what happened
order_placed = Event(
    event_type='OrderPlaced',
    payload={
        'order_id': 'ORD-001',
        'customer_id': 'USR-555',
        'items': [{'sku': 'SKU-001', 'qty': 2}],
        'total': 99.98,
    },
)

# Correct: Commands tell an agent what to do
reserve_items = Command(
    command_type='ReserveInventory',
    payload={'order_id': 'ORD-001', 'items': [{'sku': 'SKU-001', 'qty': 2}]},
    reply_to='order-agent-001',
)

print(f'Event type:   {order_placed.event_type}  (past participle ✓)')
print(f'Command type: {reserve_items.command_type}  (imperative ✓)')
print(f'Event is immutable fact; its payload should never be modified after emission.')

## 6.2 The Agent Event Bus

The in-process `AgentEventBus` demonstrates the correct semantics:
- **Idempotency**: the same `message_id` is processed at most once
- **Retries with backoff**: failed handlers are retried up to `max_retries` times
- **Dead-letter queue**: exhausted retries land in DLQ, never lost

In [ ]:
import asyncio

# Build a simple order processing pipeline using events
events_received = {
    'inventory': [], 'payment': [], 'notification': []
}

async def inventory_handler(event: Event):
    events_received['inventory'].append(event.event_type)
    print(f'[Inventory] Received: {event.event_type} for order {event.payload.get("order_id")}')

async def payment_handler(event: Event):
    events_received['payment'].append(event.event_type)
    print(f'[Payment] Received: {event.event_type}')

async def notification_handler(event: Event):
    events_received['notification'].append(event.event_type)
    print(f'[Notification] Sending confirmation for {event.payload.get("customer_id")}')

async def demo_bus():
    bus = AgentEventBus(max_retries=3)
    bus.subscribe('OrderPlaced', inventory_handler)
    bus.subscribe('OrderPlaced', payment_handler)
    bus.subscribe('OrderPlaced', notification_handler)

    # Publish event once
    event = Event(
        event_type='OrderPlaced',
        payload={'order_id': 'ORD-101', 'customer_id': 'USR-42'},
    )
    await bus.publish(event)
    print()

    # Publish the same event again - should be idempotent (skipped)
    await bus.publish(event)
    print(f'Second publish skipped (idempotency): inventory handled {len(events_received["inventory"])} time(s)')
    print()

    # Inspect subscriber counts
    print(f'Subscribers on OrderPlaced: {bus.subscriber_count("OrderPlaced")}')

asyncio.run(demo_bus())

## 6.3 Dead-Letter Queue

A handler that raises after all retries must not silently discard the event.
The DLQ captures it for later inspection, reprocessing, or alerting.

In [ ]:
import asyncio

async def demo_dlq():
    bus = AgentEventBus(max_retries=2)
    attempt_count = [0]

    async def flaky_handler(event: Event):
        attempt_count[0] += 1
        raise RuntimeError(f'Handler failed (attempt {attempt_count[0]})')

    bus.subscribe('PaymentFailed', flaky_handler)
    await bus.publish(Event(event_type='PaymentFailed', payload={'order_id': 'ORD-999'}))
    await asyncio.sleep(0.3)  # Allow retries

    dlq = bus.dead_letter_queue
    print(f'Handler attempts: {attempt_count[0]}')
    print(f'Dead-lettered messages: {len(dlq)}')
    for msg, err in dlq.drain():
        print(f'  DLQ entry: {msg.event_type} → {err}')

asyncio.run(demo_dlq())

## 6.4 Causal Tracing with EventContext

In a multi-agent system, one event may trigger a chain of downstream events.
**EventContext** propagates `trace_id` and `span_id` through the chain,
enabling reconstruction of the full causal timeline in Azure Application Insights.

In [ ]:
# Demonstrate causal chain propagation

# Root context: created at the edge (API gateway, UI event)
root_ctx = EventContext()
print(f'Root trace_id:  {root_ctx.trace_id[:12]}...')
print(f'Root span_id:   {root_ctx.span_id[:12]}...')
print()

# Child span: created when order agent starts processing
order_ctx = root_ctx.new_child()
print(f'Order span_id:  {order_ctx.span_id[:12]}...  (parent: {order_ctx.parent_span_id[:12]}...)')
print(f'Same trace_id:  {order_ctx.trace_id == root_ctx.trace_id}  # <-- links to root')
print()

# Grandchild span: inventory agent spawned by order agent
inventory_ctx = order_ctx.new_child()
print(f'Inventory span: {inventory_ctx.span_id[:12]}...  (parent: {inventory_ctx.parent_span_id[:12]}...)')
print(f'Same trace_id:  {inventory_ctx.trace_id == root_ctx.trace_id}  # entire chain traced')
print()
print('This trace_id can be searched in Azure Application Insights to see the full causal chain.')

## 6.5 Azure Service Bus in Production

Replace the in-process `AgentEventBus` with **Azure Service Bus** for production:

```python
from azure.servicebus.aio import ServiceBusClient
from azure.identity.aio import DefaultAzureCredential

# Passwordless auth (recommended)
credential = DefaultAzureCredential()
async with ServiceBusClient(
    fully_qualified_namespace='your-ns.servicebus.windows.net',
    credential=credential,
) as client:
    sender = client.get_queue_sender('agent-events')
    msg = ServiceBusMessage(json.dumps(event.__dict__, default=str))
    await sender.send_messages(msg)
```

Azure Service Bus provides at-least-once delivery, DLQ, sessions, and scheduled delivery.